In [1]:
import torch
import bintrans
import numpy as np

In [2]:
folder_name = input('输入要加载的模型所属文件夹：')

输入要加载的模型所属文件夹：model4


需要修改的部分有：  
根目录名称、缩放因子、模型名称与输入名称  
生成基于该模型与输入的ram初始化文本文件

In [2]:
filepath = '../trained_model/' + folder_name
scale = 2 ** 13

加载模型与输入数据

In [3]:
model = torch.load(filepath + 'mymodel6_79.pth')
all_x = np.load(filepath + 'all_x1_nm.npy')
print(all_x.shape)

(11788, 30, 7)


## 保存python模型的所有参数

In [4]:
parameters = {}
for name, param in model.named_parameters():
    parameters[name] = param.detach().numpy()

In [5]:
def param2txt(parameter):
    # 遍历参数的所有关键字
    for key in parameters.keys():
        #将参数乘以定点化缩放系数并展平成一维
        flo = (parameters[key].copy().T.reshape(-1)*scale).tolist()
        
        #将尾数去掉并转换成二进制补码字符串
        bnr = []
        for w in flo:
            bnr.append(bintrans.dec2bnr(int(w))) 
        
        #写入txt文件中
        filename = key.replace('.', '_') + '.txt'
        with open(filepath +  'pyparam/' + filename, 'w') as file:
            for item in bnr:
                file.write(item + '\n')

In [6]:
param2txt(parameters)

## 生成与tb格式符合的参数文件

### 权重与偏置

In [7]:
filelist1 = ['U_i1.txt', 'U_f1.txt', 'U_c1.txt', 'U_o1.txt']
filelist2 = ['U_i2.txt', 'U_f2.txt', 'U_c2.txt', 'U_o2.txt']
filelist3 = ['V_i1.txt', 'V_f1.txt', 'V_c1.txt', 'V_o1.txt']
filelist4 = ['V_i2.txt', 'V_f2.txt', 'V_c2.txt', 'V_o2.txt']
filelist5 = ['b_i1.txt', 'b_f1.txt', 'b_c1.txt', 'b_o1.txt']
filelist6 = ['b_i2.txt', 'b_f2.txt', 'b_c2.txt', 'b_o2.txt']
filelist7 = ['fc_weight.txt']
filelist8 = ['fc_bias.txt']

In [8]:
def contxt(filepath, filelist):
    j = 0
    t_new = []
    for filename in filelist:
        f = open(filepath + 'pyparam/' + filename)
        t = f.readlines()
        if j==0:
            t_new = t
        else:
            for i in range(len(t)):
                t_new[i] = t_new[i].strip('\n') + t[i]
        j = j + 1
    return t_new

In [9]:
wx1 = contxt(filepath, filelist1)
wx2 = contxt(filepath, filelist2)
wh1 = contxt(filepath, filelist3)
wh2 = contxt(filepath, filelist4)
b1 = contxt(filepath, filelist5)
b2 = contxt(filepath, filelist6)
wf = contxt(filepath, filelist7)
bf = contxt(filepath, filelist8)

In [10]:
wr_dict = {'wx1': wx1, 'wx2' : wx2, 'wh1' : wh1, 'wh2' : wh2, 'b1' : b1, 'b2' : b2, 'wf' : wf, 'bf' : bf}

In [11]:
for key in wr_dict.keys():
    savename = filepath + 'vparam/' + key + '.txt'
    with open(savename, 'w') as f:
        f.writelines(wr_dict[key])

### 自变量

In [12]:
x = all_x[0, :, :]

In [13]:
x_bnr = []
for i in range(x.shape[0]):
    for j in range(x.shape[1]):
        x_bnr.append(bintrans.flo2brn(x[i, j], scale) + '\n')

0001100011111111
0001011101001010
0000001011101011
0000001100000101
0000000010000001
0000111111100011
0001100101010001
0001100011111111
0001011101001010
0000001011101011
0000001100000101
0000000010000001
0001010011111010
0001101011000101
0001100011111111
0001011101001010
0000001011101011
0000001100000101
0000000010000001
0001010011111010
0001101011101001
0001100011111111
0001011101001010
0000001011101011
0000001100000101
0000000010000001
0001010011111010
0001101011101001
0001100011111111
0001011101001010
0000001011101011
0000001100000101
0000000010000001
0001010011111010
0001101011110111
0001100011111111
0001011101001010
0000001011101011
0000001100000101
0000000010000001
0001010011111010
0001101100001110
0001100011111111
0001011101001010
0000001011101011
0000001100000101
0000000010000001
0001010011111010
0001101100000110
0001100011111111
0001011101001010
0000001011101011
0000001100000101
0000000010000001
0001010011111010
0001101011111110
0001100011111111
0001011101001010
00000010111010

In [14]:
with open(filepath + 'vparam/' + 'x.txt', 'w') as fx:
    fx.writelines(x_bnr)

### 中间变量

In [15]:
zero = '0000000000000000\n'

In [16]:
h = [zero] * 64

In [17]:
with open(filepath + 'vparam/' + 'h.txt', 'w') as fh:
    fh.writelines(h)

### 线形层偏置

In [26]:
wr_dict['bf'][0].strip('\n')
bb = bintrans.bnr2dec(wr_dict['bf'][0].strip('\n'))
hex(bb)

'0x345'